In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# 1. Print where your script is actually looking for the .env file
print("Current Working Directory:", Path.cwd())
print("Expected .env Path:", Path.cwd() / ".env")

# 2. Explicitly pass the absolute path if the default search fails
env_path = ".env"
file_found = load_dotenv()

print(f"Was the .env file found at explicit path? {file_found}")
print("Variable value:", os.getenv("BUCKET"))


In [ ]:
import influxdb_client
from dotenv import load_dotenv
import os
from datetime import datetime, timezone, timedelta
import sys
sys.path.append('/Users/xz498/Library/CloudStorage/OneDrive-DrexelUniversity/Chang Lab - Documents/General/Individual/Xinqiao Zhang/data_analysis/ultrasonic_ml/src/')

def initialize_client():
    load_dotenv()
    
    org = os.getenv("ORG")
    token = os.getenv("API_KEY")
    url = os.getenv("DB_URL")
    
    client = influxdb_client.InfluxDBClient(
        url=url,
        token=token,
        org=org,
    )
    
    return client


def get_sensor_data(start_date: datetime, end_date: datetime, sensor: str, 
                      celsius=False, fahrenheit=False, pressure=False, humidity=False):
    
    fields_to_query = []
    if celsius:
        fields_to_query.append("celcius")
    if fahrenheit:
        fields_to_query.append("fahrenheit")
    if pressure:
        fields_to_query.append("pressure")
    if humidity:
        fields_to_query.append("humidity")

    if not fields_to_query:
        print("No data fields selected. Returning no data.")
        return []

    flux_field_filter = " or ".join([f'r._field == "{field}"' for field in fields_to_query])

    duration = end_date - start_date
    duration_in_hours = duration.total_seconds() / 3600

    window_period = None

    if duration_in_hours > 336:
        window_period = "90m"
    elif duration_in_hours > 168:
        window_period = "30m"
    elif duration_in_hours > 72:
        window_period = "10m"
    elif duration_in_hours > 24:
        window_period = "5m"

    aggregate_string = ""
    if window_period:
        aggregate_string = f'''
          |> aggregateWindow(every: {window_period}, fn: mean, createEmpty: false)
        '''
    
    start = start_date.isoformat()
    end = end_date.isoformat()

    query = f'''
        from(bucket: "{os.getenv("BUCKET")}")
          |> range(start: {start}, stop: {end})
          |> filter(fn: (r) => r.topic == "sensor/{sensor}")
          |> filter(fn: (r) => {flux_field_filter})
          {aggregate_string}
    '''
    
    client = initialize_client()
    query_api = client.query_api()
    result_tables = query_api.query(org=os.getenv("ORG"), query=query)
    client.close()

    results = []
    for table in result_tables:
        for record in table.records:
            results.append({
                "time": record.get_time().isoformat(),
                "field": record.get_field(),
                "value": record.get_value()
            })
            
    return results

In [7]:
# end_utc = datetime.now(timezone.utc)
# start_utc = end_utc - timedelta(minutes=30)

start_utc = datetime(2026, 6, 29, 14, 45, tzinfo=timezone.utc)
end_utc = datetime(2026, 6, 29, 16, 45, tzinfo=timezone.utc)

res = get_sensor_data(start_utc, end_utc, "lab", celsius=True);

ConnectTimeoutError: (<urllib3.connection.HTTPConnection object at 0x12e8a11c0>, 'Connection to 10.147.18.165 timed out. (connect timeout=20.0)')

In [18]:
res


NameError: name 'res' is not defined